### Data Preparation

In [13]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn
import torch
from copy import copy
from time import perf_counter

import psutil  # If missing, run %pip install psutil in a notebook cell.
from torch.utils.data import Subset


In [14]:
dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True
)

pixels = dataset.data
mean = pixels.mean(axis=(0,1,2)) / 255
std = pixels.std(axis=(0, 1, 2)) / 255.0

train_transform =  transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

test_transform =  transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        std=std,
        mean=mean
    ),
])

training_dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform
)

testing_dataset = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform
)

### Building a CNN from scratch

In [15]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.testingStack = nn.Sequential(
            # 3x3 x 3 x 32 + 32
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            # 3x3 x 32 x 32 + 32
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # 3x3 x 32 x 64 + 64
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # 3x3 x 64x 128 + 128
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),

            # 128 x 10 + 10
            nn.Linear(128,10)
        )

    def forward(self,x):
        logits = self.testingStack(x)
        return logits
    
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
net = NeuralNetwork().to(device)

In [16]:
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001

indices = torch.randperm(
    len(training_dataset), generator=torch.Generator().manual_seed(42)
).tolist()

eval_transform = transforms.ToTensor()
validation_dataset = copy(training_dataset)
validation_dataset.transform = eval_transform
test_dataset = copy(testing_dataset)
test_dataset.transform = eval_transform

trainloader = DataLoader(
    Subset(training_dataset, indices[5000:]), BATCH_SIZE, shuffle=True
)
validation_loader = DataLoader(
    Subset(validation_dataset, indices[:5000]), BATCH_SIZE
)
testing_dataloader = DataLoader(test_dataset, BATCH_SIZE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    net.parameters(),
    lr=LEARNING_RATE
)

In [17]:
def train(trainloader, net, criterion, optimizer):
    net.train()
    total_loss = correct = total = 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.detach().argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

@torch.inference_mode()
def evaluate(loader, net, criterion):
    net.eval()
    total_loss = correct = total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = net(inputs)

        total_loss += criterion(outputs, labels).item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [18]:
device_type = torch.device(device).type

for epoch in range(1, EPOCHS + 1):
    start = perf_counter()
    train_loss, train_acc = train(trainloader, net, criterion, optimizer)
    val_loss, val_acc = evaluate(validation_loader, net, criterion)

    if device_type == "mps":
        torch.mps.synchronize()
    elif device_type == "cuda":
        torch.cuda.synchronize(device)

    elapsed = perf_counter() - start
    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss: {train_loss:.4f} | val loss: {val_loss:.4f} | "
        f"train acc: {train_acc:.2%} | val acc: {val_acc:.2%} | "
        f"time: {elapsed:.2f}s"
    )

mib = 1024**2
print(f"After training | process RAM: {psutil.Process().memory_info().rss / mib:.1f} MiB")
if device_type == "mps":
    print(f"MPS tensor memory: {torch.mps.current_allocated_memory() / mib:.1f} MiB")
elif device_type == "cuda":
    print(f"CUDA tensor memory: {torch.cuda.memory_allocated(device) / mib:.1f} MiB")

test_loss, test_acc = evaluate(testing_dataloader, net, criterion)
print(f"Final test | loss: {test_loss:.4f} | accuracy: {test_acc:.2%}")

Epoch 1/20 | train loss: 1.8384 | val loss: 1.5708 | train acc: 29.76% | val acc: 41.96% | time: 12.64s
Epoch 2/20 | train loss: 1.4854 | val loss: 1.4040 | train acc: 45.21% | val acc: 48.58% | time: 11.16s
Epoch 3/20 | train loss: 1.3121 | val loss: 1.3589 | train acc: 52.34% | val acc: 52.40% | time: 11.05s
Epoch 4/20 | train loss: 1.2010 | val loss: 1.1667 | train acc: 57.02% | val acc: 59.12% | time: 12.10s
Epoch 5/20 | train loss: 1.1031 | val loss: 1.0570 | train acc: 60.76% | val acc: 63.24% | time: 11.57s
Epoch 6/20 | train loss: 1.0354 | val loss: 0.9590 | train acc: 63.39% | val acc: 66.42% | time: 11.38s
Epoch 7/20 | train loss: 0.9732 | val loss: 0.9866 | train acc: 65.70% | val acc: 65.90% | time: 11.23s
Epoch 8/20 | train loss: 0.9231 | val loss: 0.8829 | train acc: 67.34% | val acc: 69.22% | time: 11.31s
Epoch 9/20 | train loss: 0.8864 | val loss: 0.8551 | train acc: 68.64% | val acc: 70.10% | time: 11.48s
Epoch 10/20 | train loss: 0.8527 | val loss: 0.8509 | train acc:

In [38]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("/Users/tykea/Developer/deep-learning/cifar-10-from-scratch/model.pth",weights_only=True))
model

NeuralNetwork(
  (testingStack): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): AdaptiveAvgPool2d(output_size=1)
    (11): Flatten(start_dim=1, end_dim=-1)
    (12): Linear(in_features=128, out_features=10, bias=True)
  )
)

In [45]:
classes =['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

testing_dataset.classes

# example, label = testing_dataset[0]
# model.eval()
# # index = 23
# with torch.no_grad():
#     x = example.unsqueeze(0).to(device)
#     pred = model(x)
#     predicted, actual = classes[pred[0].argmax(0)], classes[label]
#     print(f'Predicted: "{predicted}", Actual: "{actual}"')

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']